# KẾT NỐI SQL SERVER VỚI PYTHON - WEEK 4

## Mục lục
1. **Cài đặt và thiết lập**
2. **Kết nối SQL Server**
3. **Thực hiện các thao tác CRUD**
4. **Xử lý dữ liệu từ SQL Server**

---

## Yêu cầu
- SQL Server đã được cài đặt và chạy
- Database `dbPythonTestSQL` đã được tạo
- Bảng `tblUsers` đã được tạo với cấu trúc:
  - uid (varchar): ID nhân viên
  - uname (varchar): Tên nhân viên  
  - birthdate (date): Ngày sinh
  - salary (int): Lương
  - deptid (int): ID phòng ban

---


## 1. Cài đặt và thiết lập

### 1.1. Cài đặt thư viện cần thiết


In [1]:
# Cài đặt thư viện pyodbc để kết nối SQL Server
# Chạy lệnh sau trong terminal hoặc command prompt:
# pip install pyodbc

# Import các thư viện cần thiết
import pyodbc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Đã import thành công các thư viện cần thiết!")
print("Lưu ý: Nếu chưa cài đặt pyodbc, hãy chạy: pip install pyodbc")


ModuleNotFoundError: No module named 'pyodbc'

### 1.2. Thiết lập thông tin kết nối


In [ ]:
# Thông tin kết nối SQL Server
server = 'localhost'  # hoặc IP của server SQL Server
database = 'dbPythonTestSQL'
username = 'sa'
password = '123456'
driver = '{ODBC Driver 17 for SQL Server}'  # hoặc '{SQL Server}'

# Chuỗi kết nối
connection_string = f'DRIVER={driver};SERVER={server};DATABASE={database};UID={username};PWD={password}'

print("Thông tin kết nối:")
print(f"Server: {server}")
print(f"Database: {database}")
print(f"Username: {username}")
print(f"Driver: {driver}")
print(f"Connection String: {connection_string}")

# Kiểm tra các driver ODBC có sẵn
print("\nCác driver ODBC có sẵn:")
drivers = [x for x in pyodbc.drivers() if 'SQL Server' in x]
for driver in drivers:
    print(f"- {driver}")


## 2. Kết nối SQL Server


In [ ]:
# Hàm kết nối SQL Server
def connect_to_sql_server():
    try:
        # Tạo kết nối
        conn = pyodbc.connect(connection_string)
        print("✅ Kết nối SQL Server thành công!")
        return conn
    except Exception as e:
        print(f"❌ Lỗi kết nối SQL Server: {e}")
        print("\nCác bước khắc phục:")
        print("1. Kiểm tra SQL Server đã chạy chưa")
        print("2. Kiểm tra thông tin kết nối (server, database, username, password)")
        print("3. Kiểm tra driver ODBC đã cài đặt chưa")
        print("4. Kiểm tra firewall và network")
        return None

# Thử kết nối
conn = connect_to_sql_server()

if conn:
    # Tạo cursor
    cursor = conn.cursor()
    print("✅ Đã tạo cursor thành công!")
    
    # Kiểm tra kết nối bằng cách thực hiện một query đơn giản
    try:
        cursor.execute("SELECT @@VERSION")
        version = cursor.fetchone()
        print(f"✅ Phiên bản SQL Server: {version[0][:50]}...")
    except Exception as e:
        print(f"❌ Lỗi khi thực hiện query: {e}")
else:
    print("❌ Không thể kết nối SQL Server!")


## 3. Thực hiện các thao tác CRUD

### 3.1. Tạo bảng và dữ liệu mẫu (nếu chưa có)


In [ ]:
# Tạo bảng tblUsers nếu chưa tồn tại
def create_table_if_not_exists():
    if conn:
        try:
            cursor = conn.cursor()
            
            # Kiểm tra bảng đã tồn tại chưa
            cursor.execute("""
                SELECT COUNT(*) 
                FROM INFORMATION_SCHEMA.TABLES 
                WHERE TABLE_NAME = 'tblUsers'
            """)
            
            table_exists = cursor.fetchone()[0]
            
            if table_exists == 0:
                # Tạo bảng tblUsers
                create_table_query = """
                CREATE TABLE tblUsers (
                    uid VARCHAR(10) PRIMARY KEY,
                    uname VARCHAR(50) NOT NULL,
                    birthdate DATE,
                    salary INT,
                    deptid INT
                )
                """
                cursor.execute(create_table_query)
                conn.commit()
                print("✅ Đã tạo bảng tblUsers thành công!")
                
                # Thêm dữ liệu mẫu
                sample_data = [
                    ('u0001', 'John Smith', '1990-01-15', 5000, 1),
                    ('u0002', 'Jane Doe', '1985-05-20', 6000, 2),
                    ('u0003', 'Bob Johnson', '1992-08-10', 4500, 1),
                    ('u0004', 'Alice Brown', '1988-12-03', 5500, 3),
                    ('u0005', 'Charlie Wilson', '1995-03-25', 4000, 2)
                ]
                
                insert_query = "INSERT INTO tblUsers (uid, uname, birthdate, salary, deptid) VALUES (?, ?, ?, ?, ?)"
                cursor.executemany(insert_query, sample_data)
                conn.commit()
                print("✅ Đã thêm dữ liệu mẫu thành công!")
            else:
                print("✅ Bảng tblUsers đã tồn tại!")
                
        except Exception as e:
            print(f"❌ Lỗi khi tạo bảng: {e}")
    else:
        print("❌ Không có kết nối SQL Server!")

# Thực hiện tạo bảng
create_table_if_not_exists()


### 3.2. Thực hiện các thao tác CRUD theo yêu cầu


## 4. Xử lý dữ liệu từ SQL Server


In [ ]:
# Hiển thị dữ liệu cuối cùng và thống kê
if conn:
    print("📊 DỮ LIỆU CUỐI CÙNG TRONG BẢNG:")
    print("=" * 60)
    
    try:
        # Lấy tất cả dữ liệu
        final_query = """
        SELECT uid, uname, birthdate, salary, deptid,
               DATEDIFF(YEAR, birthdate, GETDATE()) as age
        FROM tblUsers
        ORDER BY uid
        """
        df_final = pd.read_sql(final_query, conn)
        
        print("📋 Danh sách tất cả nhân viên:")
        print(df_final.to_string(index=False))
        
        # Thống kê cơ bản
        print(f"\n📈 THỐNG KÊ:")
        print(f"   - Tổng số nhân viên: {len(df_final)}")
        print(f"   - Lương trung bình: {df_final['salary'].mean():.2f}")
        print(f"   - Lương cao nhất: {df_final['salary'].max()}")
        print(f"   - Lương thấp nhất: {df_final['salary'].min()}")
        print(f"   - Tuổi trung bình: {df_final['age'].mean():.1f}")
        
        # Thống kê theo phòng ban
        print(f"\n📊 THỐNG KÊ THEO PHÒNG BAN:")
        dept_stats = df_final.groupby('deptid').agg({
            'uname': 'count',
            'salary': ['mean', 'min', 'max'],
            'age': 'mean'
        }).round(2)
        print(dept_stats)
        
        # Lưu dữ liệu vào file CSV
        df_final.to_csv('sql_server_data.csv', index=False, encoding='utf-8')
        print(f"\n💾 Đã lưu dữ liệu vào file 'sql_server_data.csv'")
        
    except Exception as e:
        print(f"❌ Lỗi: {e}")
    
    # Đóng kết nối
    conn.close()
    print(f"\n🔒 Đã đóng kết nối SQL Server!")
    
else:
    print("❌ Không có kết nối SQL Server để xử lý dữ liệu!")


## 5. Tóm tắt và kết luận

### 5.1. Các thao tác đã thực hiện
1. ✅ **Kết nối SQL Server** - Thiết lập kết nối với database
2. ✅ **Tạo bảng** - Tạo bảng `tblUsers` với cấu trúc phù hợp
3. ✅ **Thêm dữ liệu mẫu** - Thêm 5 nhân viên mẫu
4. ✅ **Truy vấn dữ liệu** - Lấy thông tin nhân viên theo điều kiện
5. ✅ **Thêm nhân viên mới** - Thêm nhân viên David (u0006)
6. ✅ **Cập nhật dữ liệu** - Cập nhật lương nhân viên u0002
7. ✅ **Xóa dữ liệu** - Xóa nhân viên u0006
8. ✅ **Thống kê dữ liệu** - Phân tích và thống kê dữ liệu

### 5.2. Các lưu ý quan trọng
- **Bảo mật**: Luôn sử dụng parameterized queries để tránh SQL injection
- **Xử lý lỗi**: Luôn có try-catch để xử lý các lỗi có thể xảy ra
- **Đóng kết nối**: Luôn đóng kết nối sau khi sử dụng xong
- **Commit**: Nhớ commit sau các thao tác INSERT, UPDATE, DELETE

### 5.3. Các thư viện cần thiết
```bash
pip install pyodbc pandas numpy
```

### 5.4. Cấu trúc bảng tblUsers
```sql
CREATE TABLE tblUsers (
    uid VARCHAR(10) PRIMARY KEY,
    uname VARCHAR(50) NOT NULL,
    birthdate DATE,
    salary INT,
    deptid INT
)
```

---
**Hoàn thành tất cả các yêu cầu kết nối SQL Server với Python!** 🎉
